# Regime Classification — Hyperparameter Research

**Goal:** Evaluate whether `RegimeClassificationModel` hyperparameters produce stable, 
separable regime states that carry information — using enriched data beyond raw OHLCV.

**Data Sources:**
1. Binance Futures OHLCV (1m base → aggregated to 30m, 1h, 4h)
2. TradingView indices (BTC.D, TOTAL2, TOTAL3) — macro/breadth context
3. L2 orderbook features (bid/ask imbalance, depth ratio, spread)
4. Multi-timeframe (MTF) alignment — same asset across TFs

**Approach:**
- Phase 1: Load data, compute regime features across param grid
- Phase 2: Measure regime **quality** (state separation, stability, transition smoothness)
- Phase 3: Optuna sweep on quality metrics
- Phase 4: Check if quality-optimized regimes carry downstream signal (IC, rank correlation)

**Key Insight:** Prior attempts failed optimizing regime → downstream Sharpe directly.
This notebook optimizes regime **quality first**, then checks if quality ↔ alpha.

In [ ]:
# ── Cell 1: Imports and Setup ──────────────────────────────────────
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime, timezone, timedelta
from binance.um_futures import UMFutures
from scipy import stats
from sklearn.metrics import calinski_harabasz_score, silhouette_score
from tqdm import tqdm

# Internal imports — pure computation, no side effects
from libs.models.regime_classification.model import RegimeClassificationModel
from libs.models.regime_classification.config import (
    RegimeClassificationConfig, BCPDConfig, HMMConfig, VolConfig,
    HilbertConfig, EWMAVolConfig, TrendStrengthConfig,
)
from libs.models.regime_classification.contracts import RegimeFeatureOutput
from libs.models.regime_classification.l2_features import compute_l2_features

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)

print("Imports OK")

## Phase 0: Data Loading

Fetch OHLCV from Binance API + TradingView index data.
MTF: load 30m, 1h, 4h for the same asset to build cross-TF features.

In [ ]:
# ── Cell 2: Binance OHLCV Fetcher ─────────────────────────────────
client = UMFutures()

def fetch_klines(symbol: str, interval: str, start_str: str, end_str: str) -> pd.DataFrame:
    """Fetch OHLCV from Binance with pagination."""
    start_ms = int(datetime.strptime(start_str, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp() * 1000)
    end_ms = int(datetime.strptime(end_str, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp() * 1000)
    
    all_rows = []
    current = start_ms
    while current < end_ms:
        klines = client.klines(symbol=symbol, interval=interval, startTime=current, endTime=end_ms, limit=1500)
        if not klines:
            break
        for k in klines:
            all_rows.append({
                "timestamp": pd.Timestamp(k[0], unit="ms", tz="UTC"),
                "open": float(k[1]), "high": float(k[2]), "low": float(k[3]),
                "close": float(k[4]), "volume": float(k[5]),
                "taker_buy_base": float(k[9]),
            })
        current = int(klines[-1][0]) + 1
    
    df = pd.DataFrame(all_rows)
    if not df.empty:
        df = df.drop_duplicates(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
    return df

# ── Configuration ──
SYMBOL = "BTCUSDT"
START = "2024-06-01"
END = "2026-06-01"
TIMEFRAMES = {"30m": "30m", "1h": "1h", "4h": "4h"}

print(f"Fetching {SYMBOL} OHLCV: {START} → {END}")
data = {}
for label, interval in TIMEFRAMES.items():
    df = fetch_klines(SYMBOL, interval, START, END)
    data[label] = df
    print(f"  {label}: {len(df)} bars, {df['timestamp'].min()} → {df['timestamp'].max()}")

# Primary working timeframe
df_1h = data["1h"].copy()
print(f"\nPrimary TF (1h): {len(df_1h)} bars")

In [ ]:
# ── Cell 3: TradingView Index Data (BTC.D, TOTAL2, TOTAL3) ────────
# Fetch TV indices from Binance-style endpoint or manual CSV
# These are stored in tv_index_ohlcv in production — here we use the API

TV_INDICES = {
    "BTCDOM": "BTCDOMUSDT",  # BTC dominance proxy on Binance
}

tv_data = {}
for label, sym in TV_INDICES.items():
    try:
        df_tv = fetch_klines(sym, "1h", START, END)
        tv_data[label] = df_tv
        print(f"TV Index {label} ({sym}): {len(df_tv)} bars")
    except Exception as e:
        print(f"TV Index {label} failed: {e} (will continue without)")

# For TOTAL2/TOTAL3 — these aren't on Binance, would come from TV scraper DB.
# In research, we proceed with what's available.
print(f"\nTV indices loaded: {list(tv_data.keys())}")

In [ ]:
# ── Cell 4: MTF Feature Alignment ─────────────────────────────────
# Align 30m, 4h data to 1h index for MTF cross-timeframe features

def align_mtf(df_base: pd.DataFrame, df_other: pd.DataFrame, suffix: str) -> pd.DataFrame:
    """Forward-fill higher/lower TF data onto the base timeframe index."""
    df_other = df_other.set_index("timestamp")[["close", "volume"]].rename(
        columns={"close": f"close_{suffix}", "volume": f"volume_{suffix}"}
    )
    df_base = df_base.set_index("timestamp") if "timestamp" in df_base.columns else df_base
    merged = df_base.join(df_other, how="left")
    merged[[f"close_{suffix}", f"volume_{suffix}"]] = merged[[f"close_{suffix}", f"volume_{suffix}"]].ffill()
    return merged.reset_index() if "timestamp" not in merged.columns else merged

df_mtf = df_1h.copy()
if "30m" in data:
    df_mtf = align_mtf(df_mtf, data["30m"], "30m")
if "4h" in data:
    df_mtf = align_mtf(df_mtf, data["4h"], "4h")
if "BTCDOM" in tv_data:
    df_mtf = align_mtf(df_mtf, tv_data["BTCDOM"], "btcdom")

# Compute MTF-derived features
if "close_4h" in df_mtf.columns:
    df_mtf["mtf_4h_trend"] = (df_mtf["close"] / df_mtf["close_4h"].rolling(20).mean() - 1).fillna(0)
    df_mtf["mtf_4h_vol_ratio"] = (df_mtf["volume"] / df_mtf["volume_4h"].rolling(20).mean()).fillna(1)
if "close_btcdom" in df_mtf.columns:
    df_mtf["btcdom_chg_20"] = df_mtf["close_btcdom"].pct_change(20).fillna(0)

print(f"MTF-aligned df: {df_mtf.shape}")
print(f"MTF columns: {[c for c in df_mtf.columns if 'mtf' in c or 'btcdom' in c]}")

## Phase 1: Baseline Regime Classification

Run the model with default hyperparameters. Inspect the output feature distribution.

In [ ]:
# ── Cell 5: Run Regime Model with Default Params ──────────────────
model_default = RegimeClassificationModel()  # all defaults from hyperparameter_schema

print("Default hyperparameters:")
for k, v in model_default.params.items():
    print(f"  {k}: {v}")

# Run batch evaluation on 1h close/volume
df_input = df_1h[["close", "volume"]].copy()
regime_series = model_default.batch_evaluate(df_input)

# Unpack into DataFrame
regime_df = pd.DataFrame(regime_series.tolist(), index=df_input.index)
print(f"\nRegime output: {regime_df.shape}")
print(f"Columns: {list(regime_df.columns)}")
print(f"\nSample (last 5 bars):")
regime_df.tail()

In [ ]:
# ── Cell 6: Regime Feature Distributions ──────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(16, 12))

feature_cols = [
    "vol_percentile", "hurst", "changepoint_prob",
    "trend_strength", "hilbert_period", "hilbert_confidence",
    "fwd_vol_ewma", "realized_vol", "cp_entropy",
]

for ax, col in zip(axes.flat, feature_cols):
    vals = regime_df[col].dropna()
    ax.hist(vals, bins=50, alpha=0.7, edgecolor='white', linewidth=0.3)
    ax.set_title(f"{col}\nμ={vals.mean():.3f}  σ={vals.std():.3f}", fontsize=10)
    ax.axvline(vals.median(), color='red', linestyle='--', alpha=0.7)

plt.suptitle(f"Regime Feature Distributions — {SYMBOL} 1h (default params)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 7: HMM State Posteriors Over Time ────────────────────────
hmm_cols = [c for c in regime_df.columns if c.startswith("hmm_p_state_")]
n_states = int(regime_df["hmm_n_states"].mode()[0]) if "hmm_n_states" in regime_df else 2

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# Price
axes[0].plot(df_1h["close"].values, linewidth=0.5, alpha=0.8)
axes[0].set_ylabel("Close")
axes[0].set_title(f"{SYMBOL} 1h — HMM Regime Posteriors (default params)")

# Posteriors stacked
for i, col in enumerate(hmm_cols[:n_states]):
    axes[1].fill_between(range(len(regime_df)), regime_df[col], alpha=0.5, label=f"State {i}")
axes[1].set_ylabel("Posterior P")
axes[1].legend(loc="upper right")

# Changepoint probability
axes[2].fill_between(range(len(regime_df)), regime_df["changepoint_prob"], alpha=0.6, color="red")
axes[2].set_ylabel("CP Prob")
axes[2].set_xlabel("Bar Index")

plt.tight_layout()
plt.show()

## Phase 2: Regime Quality Metrics

Define quality metrics that measure whether regimes are **real** (separable, stable, informative) 
rather than noise. These are the optimization objectives.

**Metrics:**
1. **State Separation** — Calinski-Harabasz score on returns conditioned by HMM hard state
2. **Transition Stability** — average regime persistence (bars in same state before switching)
3. **Conditional Return Spread** — |mean_return(state_i) - mean_return(state_j)| across states
4. **Hurst-Return Correlation** — does Hurst actually predict mean-reversion vs trending?
5. **Vol Percentile Calibration** — does high vol_percentile predict future realized vol?

In [ ]:
# ── Cell 8: Regime Quality Scoring Functions ──────────────────────

def compute_regime_quality(regime_df: pd.DataFrame, price_df: pd.DataFrame) -> dict:
    """Compute regime quality metrics from model output and price data.
    
    Returns a dict of metric_name → value. Higher is better for all metrics.
    """
    metrics = {}
    n = len(regime_df)
    
    # Forward 1-bar log returns
    returns = np.log(price_df["close"] / price_df["close"].shift(1)).values
    returns[0] = 0  # fill first bar
    
    # Forward 10-bar returns (for regime predictive power)
    fwd_10 = price_df["close"].pct_change(10).shift(-10).values
    
    # --- 1. HMM State Separation (Calinski-Harabasz) ---
    hmm_cols = [c for c in regime_df.columns if c.startswith("hmm_p_state_")]
    if len(hmm_cols) >= 2:
        hard_state = regime_df[hmm_cols].values.argmax(axis=1)
        # Need at least 2 unique states for CH score
        unique_states = np.unique(hard_state)
        if len(unique_states) >= 2:
            # Feature matrix: [return, abs_return, volume_change]
            vol_change = np.log(price_df["volume"] / price_df["volume"].shift(1)).fillna(0).values
            X = np.column_stack([returns, np.abs(returns), vol_change])
            valid = ~np.isnan(X).any(axis=1) & np.isfinite(X).all(axis=1)
            if valid.sum() > len(unique_states):
                try:
                    metrics["ch_score"] = calinski_harabasz_score(X[valid], hard_state[valid])
                except Exception:
                    metrics["ch_score"] = 0.0
            else:
                metrics["ch_score"] = 0.0
        else:
            metrics["ch_score"] = 0.0
    else:
        metrics["ch_score"] = 0.0
    
    # --- 2. Transition Stability (avg bars in same state) ---
    if len(hmm_cols) >= 2:
        state_changes = np.diff(hard_state) != 0
        runs = np.split(np.arange(len(hard_state)), np.where(state_changes)[0] + 1)
        run_lengths = [len(r) for r in runs if len(r) > 0]
        metrics["avg_run_length"] = np.mean(run_lengths) if run_lengths else 1.0
        metrics["median_run_length"] = np.median(run_lengths) if run_lengths else 1.0
    else:
        metrics["avg_run_length"] = 1.0
        metrics["median_run_length"] = 1.0
    
    # --- 3. Conditional Return Spread ---
    if len(hmm_cols) >= 2 and len(unique_states) >= 2:
        state_returns = {}
        for s in unique_states:
            mask = hard_state == s
            if mask.sum() > 10:
                state_returns[s] = np.nanmean(returns[mask])
        if len(state_returns) >= 2:
            spreads = []
            states_list = list(state_returns.keys())
            for i in range(len(states_list)):
                for j in range(i + 1, len(states_list)):
                    spreads.append(abs(state_returns[states_list[i]] - state_returns[states_list[j]]))
            metrics["return_spread"] = np.mean(spreads) * 1e4  # in bps
        else:
            metrics["return_spread"] = 0.0
    else:
        metrics["return_spread"] = 0.0
    
    # --- 4. Hurst-Return Rank Correlation ---
    hurst = regime_df.get("hurst", pd.Series(np.full(n, 0.5)))
    valid = ~np.isnan(fwd_10) & ~np.isnan(hurst.values) & np.isfinite(fwd_10)
    if valid.sum() > 50:
        # Hurst < 0.5 → mean-reverting, > 0.5 → trending
        # abs(fwd_return) should be higher when hurst > 0.5 (trending)
        rho, _ = stats.spearmanr(hurst.values[valid], np.abs(fwd_10[valid]))
        metrics["hurst_fwd_corr"] = abs(rho)  # magnitude matters
    else:
        metrics["hurst_fwd_corr"] = 0.0
    
    # --- 5. Vol Percentile Calibration ---
    vol_pct = regime_df.get("vol_percentile", pd.Series(np.full(n, 50.0)))
    fwd_vol_5 = price_df["close"].pct_change().rolling(5).std().shift(-5).values
    valid = ~np.isnan(fwd_vol_5) & ~np.isnan(vol_pct.values) & np.isfinite(fwd_vol_5)
    if valid.sum() > 50:
        rho, _ = stats.spearmanr(vol_pct.values[valid], fwd_vol_5[valid])
        metrics["vol_calibration"] = rho  # should be positive
    else:
        metrics["vol_calibration"] = 0.0
    
    # --- 6. Composite quality score ---
    # Weighted blend of normalized metrics
    metrics["composite_quality"] = (
        0.25 * min(metrics["ch_score"] / 100, 1.0)  # normalize CH
        + 0.20 * min(metrics["avg_run_length"] / 20, 1.0)  # stability
        + 0.20 * min(metrics["return_spread"] / 5.0, 1.0)  # separation bps
        + 0.20 * metrics["hurst_fwd_corr"]  # predictive power
        + 0.15 * max(metrics["vol_calibration"], 0.0)  # vol calibration
    )
    
    return metrics

# Run on default params
quality_default = compute_regime_quality(regime_df, df_1h)
print("Regime Quality — Default Params:")
for k, v in quality_default.items():
    print(f"  {k}: {v:.4f}")

## Phase 3: Hyperparameter Sensitivity Analysis

Before Optuna, do a structured grid scan over the 5 exposed hyperparameters 
to understand the sensitivity landscape.

In [ ]:
# ── Cell 9: One-at-a-time Sensitivity Scan ────────────────────────

PARAM_GRIDS = {
    "bcpd_hazard_lambda": [50, 100, 150, 200, 300, 500],
    "hurst_lookback": [30, 50, 100, 150, 200, 300],
    "hmm_student_df": [2.5, 5.0, 7.5, 10.0, 15.0, 20.0],
    "hilbert_min_period": [5, 8, 10, 12, 15, 20],
    "hilbert_max_period": [20, 30, 40, 50, 60, 80],
}

defaults = {k: v.default for k, v in RegimeClassificationModel.meta.hyperparameter_schema.items()}
sensitivity_results = []

for param_name, values in tqdm(PARAM_GRIDS.items(), desc="Params"):
    for val in tqdm(values, desc=f"  {param_name}", leave=False):
        params = dict(defaults)
        params[param_name] = val
        
        # Enforce hilbert constraint
        if params["hilbert_min_period"] >= params["hilbert_max_period"]:
            continue
        
        try:
            model = RegimeClassificationModel(params)
            regime_out = model.batch_evaluate(df_input)
            regime_scan_df = pd.DataFrame(regime_out.tolist(), index=df_input.index)
            quality = compute_regime_quality(regime_scan_df, df_1h)
            
            sensitivity_results.append({
                "param": param_name,
                "value": val,
                **quality,
            })
        except Exception as e:
            print(f"  {param_name}={val} failed: {e}")

sens_df = pd.DataFrame(sensitivity_results)
print(f"\nSensitivity scan complete: {len(sens_df)} configurations")

In [ ]:
# ── Cell 10: Sensitivity Plots ────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

metric = "composite_quality"
for idx, param_name in enumerate(PARAM_GRIDS.keys()):
    ax = axes.flat[idx]
    subset = sens_df[sens_df["param"] == param_name]
    ax.plot(subset["value"], subset[metric], "o-", markersize=6)
    ax.set_xlabel(param_name)
    ax.set_ylabel(metric)
    ax.set_title(f"Sensitivity: {param_name}")
    
    # Mark default
    default_val = defaults[param_name]
    ax.axvline(default_val, color="red", linestyle="--", alpha=0.5, label=f"default={default_val}")
    ax.legend(fontsize=8)

axes.flat[-1].axis("off")
plt.suptitle(f"Hyperparameter Sensitivity — {SYMBOL} 1h, composite_quality", fontsize=14)
plt.tight_layout()
plt.show()

## Phase 3b: Hardcoded Kernel Params — Are They Worth Tuning?

The model has ~20 additional params frozen in `config.py` that never get
searched. Test the most impactful ones: `retrain_window`, `hmm_crisis_vol_mult`,
`vol_lookback`, `trend_lookback`.

In [ ]:
# ── Cell 11: Extended Param Scan (Kernel Configs) ──────────────────

def build_model_with_kernel_overrides(
    params: dict,
    retrain_window: int = 500,
    hmm_crisis_vol_mult: float = 2.0,
    vol_lookback: int = 168,
    vol_rank_window: int = 1000,
    trend_lookback: int = 20,
    ewma_decay: float = 0.94,
) -> RegimeClassificationModel:
    """Create model with custom kernel config overrides."""
    model = RegimeClassificationModel(params)
    # Override frozen config by rebuilding internals
    model._cfg = RegimeClassificationConfig(
        bcpd=BCPDConfig(hazard_lambda=params.get("bcpd_hazard_lambda", 150.0)),
        hmm=HMMConfig(
            retrain_window=retrain_window,
            hmm_student_df=params.get("hmm_student_df", 5.0),
            hurst_lookback=params.get("hurst_lookback", 100),
            hmm_crisis_vol_mult=hmm_crisis_vol_mult,
        ),
        vol=VolConfig(lookback=vol_lookback, rank_window=vol_rank_window),
        hilbert=HilbertConfig(
            min_period=params.get("hilbert_min_period", 10),
            max_period=params.get("hilbert_max_period", 40),
        ),
        ewma_vol=EWMAVolConfig(decay_factor=ewma_decay),
        trend=TrendStrengthConfig(lookback=trend_lookback),
    )
    # Rebuild vol kernel with new config
    from libs.models.regime_classification.kernels.vol_percentile import VolPercentile, VolPercentileConfig
    model._vol = VolPercentile(VolPercentileConfig(lookback=vol_lookback, rank_window=vol_rank_window))
    return model

KERNEL_GRIDS = {
    "retrain_window": [200, 300, 500, 750, 1000],
    "hmm_crisis_vol_mult": [1.0, 1.5, 2.0, 2.5, 3.0],
    "vol_lookback": [72, 168, 336, 500],
    "trend_lookback": [10, 20, 40, 60],
}

kernel_results = []
for kparam, kvals in tqdm(KERNEL_GRIDS.items(), desc="Kernel params"):
    for kv in tqdm(kvals, desc=f"  {kparam}", leave=False):
        kwargs = {kparam: kv}
        try:
            model = build_model_with_kernel_overrides(defaults, **kwargs)
            regime_out = model.batch_evaluate(df_input)
            regime_k_df = pd.DataFrame(regime_out.tolist(), index=df_input.index)
            quality = compute_regime_quality(regime_k_df, df_1h)
            kernel_results.append({"param": kparam, "value": kv, **quality})
        except Exception as e:
            print(f"  {kparam}={kv} failed: {e}")

kernel_sens_df = pd.DataFrame(kernel_results)
print(f"\nKernel param scan: {len(kernel_sens_df)} configurations")

In [ ]:
# ── Cell 12: Kernel Sensitivity Plots ─────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metric = "composite_quality"

for idx, kparam in enumerate(KERNEL_GRIDS.keys()):
    ax = axes.flat[idx]
    subset = kernel_sens_df[kernel_sens_df["param"] == kparam]
    ax.plot(subset["value"], subset[metric], "s-", markersize=6, color="cyan")
    ax.set_xlabel(kparam)
    ax.set_ylabel(metric)
    ax.set_title(f"Kernel Sensitivity: {kparam}")

plt.suptitle(f"Kernel Param Sensitivity — {SYMBOL} 1h", fontsize=14)
plt.tight_layout()
plt.show()

## Phase 4: Optuna Hyperparameter Optimization

Two-stage objective:
1. **Stage 1 (quality gate)**: composite_quality must exceed baseline
2. **Regime-quality composite** as the maximization objective

Search space: 5 exposed params + top kernel params identified from sensitivity analysis.

In [ ]:
# ── Cell 13: Optuna Regime Quality Optimization ───────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 150
STUDY_NAME = f"regime_quality_{SYMBOL}_1h"

def objective(trial: optuna.Trial) -> float:
    """Optuna objective: maximize composite regime quality."""
    params = {
        "bcpd_hazard_lambda": trial.suggest_float("bcpd_hazard_lambda", 50.0, 500.0, step=10.0),
        "hurst_lookback": trial.suggest_int("hurst_lookback", 30, 300, step=10),
        "hmm_student_df": trial.suggest_float("hmm_student_df", 2.5, 20.0, step=0.5),
        "hilbert_min_period": trial.suggest_int("hilbert_min_period", 5, 20),
        "hilbert_max_period": trial.suggest_int("hilbert_max_period", 25, 80, step=5),
    }
    
    # Enforce constraint
    if params["hilbert_min_period"] >= params["hilbert_max_period"]:
        return 0.0
    
    # Kernel params (conditionally searched based on sensitivity results)
    retrain_window = trial.suggest_int("retrain_window", 200, 1000, step=50)
    vol_lookback = trial.suggest_int("vol_lookback", 72, 500, step=24)
    trend_lookback = trial.suggest_int("trend_lookback", 10, 60, step=5)
    
    try:
        model = build_model_with_kernel_overrides(
            params,
            retrain_window=retrain_window,
            vol_lookback=vol_lookback,
            trend_lookback=trend_lookback,
        )
        regime_out = model.batch_evaluate(df_input)
        regime_trial_df = pd.DataFrame(regime_out.tolist(), index=df_input.index)
        quality = compute_regime_quality(regime_trial_df, df_1h)
        return quality["composite_quality"]
    except Exception:
        return 0.0

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\nBest trial: {study.best_trial.number}")
print(f"Best composite_quality: {study.best_value:.4f}")
print(f"Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
# ── Cell 14: Optuna Visualization ─────────────────────────────────
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_slice,
)

fig1 = plot_optimization_history(study)
plt.title(f"Optimization History — {STUDY_NAME}")
plt.tight_layout()
plt.show()

fig2 = plot_param_importances(study)
plt.title("Parameter Importances (fANOVA)")
plt.tight_layout()
plt.show()

fig3 = plot_slice(study)
plt.tight_layout()
plt.show()

## Phase 5: Validate Optimized Params

Run the optimized model and compare quality metrics vs default.
Check if quality improvement translates to any downstream signal.

In [ ]:
# ── Cell 15: Compare Default vs Optimized ─────────────────────────
best = study.best_params
best_exposed = {k: best[k] for k in defaults.keys()}

model_best = build_model_with_kernel_overrides(
    best_exposed,
    retrain_window=best.get("retrain_window", 500),
    vol_lookback=best.get("vol_lookback", 168),
    trend_lookback=best.get("trend_lookback", 20),
)

regime_best = model_best.batch_evaluate(df_input)
regime_best_df = pd.DataFrame(regime_best.tolist(), index=df_input.index)
quality_best = compute_regime_quality(regime_best_df, df_1h)

print("=" * 60)
print(f"{'Metric':<25} {'Default':>10} {'Optimized':>10} {'Delta':>10}")
print("=" * 60)
for k in quality_default:
    d = quality_default[k]
    b = quality_best[k]
    delta = b - d
    marker = "✓" if delta > 0 else "✗"
    print(f"{k:<25} {d:>10.4f} {b:>10.4f} {delta:>+10.4f} {marker}")

In [ ]:
# ── Cell 16: Downstream Signal Check (IC on forward returns) ──────
# Does the optimized regime carry forward-return information?

returns_1bar = np.log(df_1h["close"] / df_1h["close"].shift(1)).values
fwd_5 = df_1h["close"].pct_change(5).shift(-5).values
fwd_20 = df_1h["close"].pct_change(20).shift(-20).values

features_to_test = [
    "vol_percentile", "hurst", "changepoint_prob",
    "trend_strength", "hmm_crisis_prob", "hmm_transition_prob",
]

print(f"{'Feature':<25} {'IC(fwd_5)':>10} {'IC(fwd_20)':>10} {'|IC| avg':>10}")
print("-" * 60)
for feat in features_to_test:
    if feat not in regime_best_df.columns:
        continue
    vals = regime_best_df[feat].values
    
    # IC = rank correlation with forward returns
    valid_5 = ~np.isnan(fwd_5) & ~np.isnan(vals) & np.isfinite(vals)
    valid_20 = ~np.isnan(fwd_20) & ~np.isnan(vals) & np.isfinite(vals)
    
    ic_5 = stats.spearmanr(vals[valid_5], fwd_5[valid_5])[0] if valid_5.sum() > 50 else 0.0
    ic_20 = stats.spearmanr(vals[valid_20], fwd_20[valid_20])[0] if valid_20.sum() > 50 else 0.0
    avg_ic = (abs(ic_5) + abs(ic_20)) / 2
    
    marker = "★" if avg_ic > 0.03 else ""
    print(f"{feat:<25} {ic_5:>10.4f} {ic_20:>10.4f} {avg_ic:>10.4f} {marker}")

In [ ]:
# ── Cell 17: MTF-Enriched Regime Check ────────────────────────────
# Does adding MTF features change IC? (data-backed MTF suggestion test)

if "mtf_4h_trend" in df_mtf.columns:
    # Cross regime state with MTF alignment
    hmm_cols = [c for c in regime_best_df.columns if c.startswith("hmm_p_state_")]
    hard_state = regime_best_df[hmm_cols].values.argmax(axis=1)
    
    # Interaction: regime state × MTF 4h trend alignment
    mtf_trend = df_mtf["mtf_4h_trend"].values[:len(hard_state)]
    aligned = (hard_state == 0) & (mtf_trend > 0)  # trending state + 4h uptrend
    misaligned = (hard_state == 0) & (mtf_trend < 0)  # trending state but 4h downtrend
    
    fwd_10 = df_1h["close"].pct_change(10).shift(-10).values
    
    if aligned.sum() > 30 and misaligned.sum() > 30:
        ret_aligned = np.nanmean(fwd_10[aligned])
        ret_misaligned = np.nanmean(fwd_10[misaligned])
        print(f"MTF-Aligned (trending + 4h up):     fwd_10 = {ret_aligned*100:.3f}%  (n={aligned.sum()})")
        print(f"MTF-Misaligned (trending + 4h down): fwd_10 = {ret_misaligned*100:.3f}%  (n={misaligned.sum()})")
        print(f"Spread: {(ret_aligned - ret_misaligned)*100:.3f}%")
        
        # Statistical significance
        t_stat, p_val = stats.ttest_ind(
            fwd_10[aligned & ~np.isnan(fwd_10)],
            fwd_10[misaligned & ~np.isnan(fwd_10)],
        )
        print(f"t-stat: {t_stat:.3f}, p-value: {p_val:.4f}")
    else:
        print("Not enough aligned/misaligned samples for MTF test")
else:
    print("MTF columns not available — skipping MTF analysis")

In [ ]:
# ── Cell 19: Summary & Next Steps ─────────────────────────────────

print("=" * 70)
print("REGIME CLASSIFICATION HYPERPARAMETER RESEARCH — SUMMARY")
print("=" * 70)
print(f"\nAsset: {SYMBOL}, Timeframe: 1h")
print(f"Data window: {START} → {END}")
print(f"Optuna trials: {N_TRIALS}")
print(f"\nDefault composite_quality: {quality_default['composite_quality']:.4f}")
print(f"Best composite_quality:    {quality_best['composite_quality']:.4f}")
print(f"Improvement:               {quality_best['composite_quality'] - quality_default['composite_quality']:+.4f}")

print(f"\nOptimized params:")
for k, v in study.best_params.items():
    default_v = defaults.get(k, "N/A")
    print(f"  {k}: {v}  (default: {default_v})")

print(f"\nTop 3 most important params (fANOVA):")
importances = optuna.importance.get_param_importances(study)
for i, (k, v) in enumerate(list(importances.items())[:3]):
    print(f"  {i+1}. {k}: {v:.3f}")

print(f"\n--- DECISION GATE ---")
if quality_best["composite_quality"] > 0.40:
    print("✓ Regime quality above threshold — worth further investigation.")
    print("  Next: run walk-forward validation, check stability across market regimes.")
else:
    print("✗ Regime quality below threshold — confirm prior abandon decision.")
    print("  The OHLCV feature space may lack regime signal at this timeframe.")
    print("  Consider: longer TFs (4h, 1d), external data (funding, OI), or circuit-breaker approach.")